# 05 · Análisis de corridas de Optuna

Selección de hiperparámetros y desempeño por clase.
Requiere haber corrido antes `python -m src.models.tune`.

In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import sys; sys.path.insert(0, "..")
from pathlib import Path

import optuna
import pandas as pd
import matplotlib.pyplot as plt

# descubre las corridas de optuna (una carpeta por timestamp) y toma la última
OPTUNA_DIR = Path("../models/optuna")
runs = sorted(p for p in OPTUNA_DIR.iterdir() if (p / "study.db").exists())
print("corridas disponibles:", [p.name for p in runs])

run_dir = runs[-1]   # <- cambiá el índice para analizar otra corrida
study = optuna.load_study(study_name=run_dir.name, storage=f"sqlite:///{run_dir / 'study.db'}")
print(f"\ncorrida {run_dir.name}: {len(study.trials)} trials · mejor macro-F1 {study.best_value:.3f}")

## Mejores hiperparámetros

In [ ]:
pd.Series(study.best_params, name="valor").to_frame()

## Modelos guardados en la corrida (uno por trial)

In [ ]:
for p in sorted(run_dir.glob("trial_*.pt")):
    print(p.name)
print("\nmejor ->", (run_dir / "best.pt").name if (run_dir / "best.pt").exists() else "sin best.pt")

## Top 10 corridas

In [ ]:
df = study.trials_dataframe(attrs=("number", "value", "state", "params"))
df = df[df["state"] == "COMPLETE"].sort_values("value", ascending=False)
df.head(10)

## Historia de optimización — ¿va mejorando la búsqueda?

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history
plot_optimization_history(study); plt.tight_layout(); plt.show()

## Importancia de hiperparámetros — ¿cuáles pesan más?

In [ ]:
from optuna.visualization.matplotlib import plot_param_importances
plot_param_importances(study); plt.tight_layout(); plt.show()

## ¿Qué valores funcionan mejor? (slice de los más relevantes)

In [ ]:
from optuna.visualization.matplotlib import plot_slice
plot_slice(study, params=["backbone", "lr", "unfreeze_blocks", "dropout", "loss"])
plt.tight_layout(); plt.show()

## Desempeño por clase del mejor modelo (val)

Foco en las clases minoritarias (**MEL**, **SCC**). Evalúa `models/model.pt`
(el modelo final; recordá reentrenar con los mejores hiperparámetros antes).

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.models.predict import load_bundle
from src.models.dataset import MultimodalDataset
from src.data.dataset_utils import CLASSES

model, _, _ = load_bundle()
model.eval()

val = MultimodalDataset("val")
loader = DataLoader(val, batch_size=64)
ys, ps = [], []
with torch.no_grad():
    for img, tab, lab in loader:
        ps.append(model(img, tab).argmax(1))
        ys.append(lab)
y_true = torch.cat(ys).numpy()
y_pred = torch.cat(ps).numpy()
print(f"{len(y_true)} muestras de val evaluadas")

### Métricas por clase (precision / recall / F1)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

### Matriz de confusión

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicho"); plt.ylabel("Real"); plt.title("Matriz de confusión (val)")
plt.tight_layout(); plt.show()

### Recall de clases malignas

Indicador clínico clave: qué proporción de las malignas (BCC / MEL / SCC) detecta el modelo.

In [ ]:
from sklearn.metrics import recall_score

idx = {c: i for i, c in enumerate(CLASSES)}
rec = recall_score(y_true, y_pred, average=None)
for c in CLASSES:
    marca = "   <- maligna" if c in {"BCC", "MEL", "SCC"} else ""
    print(f"{c}: recall {rec[idx[c]]:.3f}{marca}")